In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# 1. Parámetros físicos y orbitales

# Constantes terrestres
mu_T = 398600.435507 # km^3/s^2, parámetro gravitacional estándar de la Tierra
R_T = 6378.1366 # km, radio ecuatorial terrestre

# Órbita inicial LEO y órbita final GEO
h_LEO = 300.0 # km, altitud de la órbita baja
r1 = R_T + h_LEO # km, radio de LEO
r2 = 42164.0 # km, radio aproximado de GEO

a_t = (r1 + r2) / 2 # Semieje mayor de la elipse de transferencia

In [ ]:
# 2. Velocidades circulares y velocidades de transferencia

# Velocidades circulares
v_c1 = np.sqrt(mu_T / r1)
v_c2 = np.sqrt(mu_T / r2)

# Velocidades en la elipse de transferencia usando vis-viva
v_t1 = np.sqrt(mu_T * (2 / r1 - 1 / a_t)) # periapsis
v_t2 = np.sqrt(mu_T * (2 / r2 - 1 / a_t)) # apoapsis

# Impulsos
delta_v1 = v_t1 - v_c1
delta_v2 = v_c2 - v_t2
delta_v_total = delta_v1 + delta_v2

# Tiempo de transferencia: media órbita de la elipse
t_transfer_s = np.pi * np.sqrt(a_t**3 / mu_T)
t_transfer_h = t_transfer_s / 3600

In [ ]:
# 3. Tabla de resultados

resultados = pd.DataFrame({
    "Magnitud": ["r1", "r2", "a_t", "v_c1", "v_t1", "Δv1", "v_t2", "v_c2", "Δv2", "Δv_total", "t_transfer"],
    "Valor": [r1, r2, a_t, v_c1, v_t1, delta_v1, v_t2, v_c2, delta_v2, delta_v_total, t_transfer_h],
    "Unidad": ["km", "km", "km", "km/s", "km/s", "km/s", "km/s", "km/s", "km/s", "km/s", "h"]
})

resultados["Valor"] = resultados["Valor"].round(4)
resultados

In [ ]:
# 4. Trayectoria teórica de Hohmann

# La elipse de transferencia tiene: periapsis = r1, apoapsis = r2, semieje mayor = a_t, excentricidad = (r2 - r1)/(r2 + r1)

e_t = (r2 - r1) / (r2 + r1)
p_t = a_t * (1 - e_t**2)

theta = np.linspace(0, np.pi, 1000) # Ángulo verdadero:de 0 a pi para recorrer media elipse
r_hohmann = p_t / (1 + e_t * np.cos(theta)) # Ecuación polar de la elipse con periapsis en theta = 0

# Conversión a coordenadas cartesianas
x_hohmann = r_hohmann * np.cos(theta)
y_hohmann = r_hohmann * np.sin(theta)

# Órbitas circulares de referencia
theta_circ = np.linspace(0, 2*np.pi, 1000)

x_leo = r1 * np.cos(theta_circ)
y_leo = r1 * np.sin(theta_circ)
x_geo = r2 * np.cos(theta_circ)
y_geo = r2 * np.sin(theta_circ)

print(f"Excentricidad de la elipse de transferencia: {e_t:.4f}")
print(f"Semilatus rectum p_t: {p_t:.4f} km")
print(f"Radio inicial de la elipse: {r_hohmann[0]:.4f} km")
print(f"Radio final de la elipse: {r_hohmann[-1]:.4f} km")

In [ ]:
# 5. Gráfica de la trayectoria teórica de Hohmann

fig, ejes = plt.subplots(1, 2, figsize=(14, 6))

# Vista global LEO-GEO
ax = ejes[0]
ax.plot(x_leo, y_leo, label="Órbita inicial LEO")
ax.plot(x_geo, y_geo, label="Órbita final GEO")
ax.plot(x_hohmann, y_hohmann, linewidth=2.5, label="Transferencia de Hohmann", color='green')
tierra = plt.Circle((0, 0), R_T, fill=False, linewidth=1.5, label="Tierra")
ax.add_patch(tierra)
ax.scatter([r1], [0], s=60, label=r"Primer impulso $\Delta v_1$")
ax.scatter([-r2], [0], s=60, label=r"Segundo impulso $\Delta v_2$")
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("x (km)")
ax.set_ylabel("y (km)")
ax.set_title("Vista global LEO-GEO")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)

# Zoom cerca de la Tierra
ax = ejes[1]
ax.plot(x_leo, y_leo, label="Órbita inicial LEO")
ax.plot(x_hohmann, y_hohmann, linewidth=2.5, label="Transferencia de Hohmann", color='green')
tierra_zoom = plt.Circle((0, 0), R_T, fill=False, linewidth=1.5, label="Tierra")
ax.add_patch(tierra_zoom)
ax.scatter([r1], [0], s=70, label=r"Primer impulso $\Delta v_1$")
lim_zoom = 9000
ax.set_xlim(-lim_zoom, lim_zoom)
ax.set_ylim(-lim_zoom, lim_zoom)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("x (km)")
ax.set_ylabel("y (km)")
ax.set_title("Zoom en Tierra y LEO")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)

plt.suptitle("Transferencia ideal de Hohmann LEO-GEO", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 6. Integración numérica con RK4

def derivadas_estado(estado, mu):
    x, y, vx, vy = estado
    r = np.sqrt(x**2 + y**2)

    ax = -mu * x / r**3
    ay = -mu * y / r**3

    return np.array([vx, vy, ax, ay])

def paso_rk4(estado, dt, mu):
    k1 = derivadas_estado(estado, mu)
    k2 = derivadas_estado(estado + 0.5 * dt * k1, mu)
    k3 = derivadas_estado(estado + 0.5 * dt * k2, mu)
    k4 = derivadas_estado(estado + dt * k3, mu)

    return estado + (dt / 6) * (k1 + 2*k2 + 2*k3 + k4)

estado_inicial = np.array([r1, 0.0, 0.0, v_t1]) # Condición inicial ideal de Hohmann

# Tiempo de integración
n_pasos = 5000
tiempos = np.linspace(0, t_transfer_s, n_pasos)
dt = tiempos[1] - tiempos[0]

# Guardar trayectoria
trayectoria = np.zeros((n_pasos, 4))
trayectoria[0] = estado_inicial

# Integración
for i in range(1, n_pasos):
    trayectoria[i] = paso_rk4(trayectoria[i-1], dt, mu_T)

# Extraer posición y velocidad
x_num = trayectoria[:, 0]
y_num = trayectoria[:, 1]
vx_num = trayectoria[:, 2]
vy_num = trayectoria[:, 3]

# Verificación del radio final
r_final = np.sqrt(x_num[-1]**2 + y_num[-1]**2)
error_radio = r_final - r2
error_relativo = 100 * error_radio / r2

print(f"Radio final numérico: {r_final:.4f} km")
print(f"Radio objetivo GEO: {r2:.4f} km")
print(f"Error absoluto: {error_radio:.6f} km")
print(f"Error relativo: {error_relativo:.6f} %")
print(f"Posición final: x = {x_num[-1]:.4f} km, y = {y_num[-1]:.4f} km")

In [ ]:
# 7. Comparación entre trayectoria teórica y trayectoria numérica

fig, ejes = plt.subplots(1, 2, figsize=(14, 6))

# Vista global
ax = ejes[0]
ax.plot(x_leo, y_leo, label="Órbita inicial LEO")
ax.plot(x_geo, y_geo, label="Órbita final GEO")
ax.plot(x_hohmann, y_hohmann, linewidth=2.5, label="Hohmann teórica")
ax.plot(x_num, y_num, "--", linewidth=2, label="Simulación RK4")

tierra = plt.Circle((0, 0), R_T, fill=False, linewidth=1.5, label="Tierra")
ax.add_patch(tierra)

ax.scatter([r1], [0], s=60, label=r"Periapsis / $\Delta v_1$")
ax.scatter([-r2], [0], s=60, label=r"Apoapsis / $\Delta v_2$")

ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("x (km)")
ax.set_ylabel("y (km)")
ax.set_title("Trayectoria teórica vs. simulación RK4")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8, loc="center right")

# Zoom/superposición de trayectorias
ax = ejes[1]
ax.plot(x_hohmann, y_hohmann, linewidth=2.5, label="Hohmann teórica")
ax.plot(x_num, y_num, "--", linewidth=2, label="Simulación RK4")

ax.scatter([r1], [0], s=60, label="Periapsis")
ax.scatter([-r2], [0], s=60, label="Apoapsis")

ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("x (km)")
ax.set_ylabel("y (km)")
ax.set_title("Superposición de trayectorias")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8, loc="lower center")

plt.suptitle("Validación computacional de la transferencia de Hohmann", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 8. Condición inicial alternativa

# Caso alternativo: velocidad tangencial 2% menor que la velocidad ideal de Hohmann
factor_alternativo = 0.98
v_alt = factor_alternativo * v_t1

estado_inicial_alt = np.array([r1, 0.0, 0.0, v_alt])

trayectoria_alt = np.zeros((n_pasos, 4))
trayectoria_alt[0] = estado_inicial_alt

for i in range(1, n_pasos):
    trayectoria_alt[i] = paso_rk4(trayectoria_alt[i-1], dt, mu_T)

x_alt = trayectoria_alt[:, 0]
y_alt = trayectoria_alt[:, 1]
vx_alt = trayectoria_alt[:, 2]
vy_alt = trayectoria_alt[:, 3]

r_final_alt = np.sqrt(x_alt[-1]**2 + y_alt[-1]**2)
error_radio_alt = r_final_alt - r2
error_relativo_alt = 100 * error_radio_alt / r2

print(f"Velocidad ideal en periapsis: {v_t1:.4f} km/s")
print(f"Velocidad alternativa: {v_alt:.4f} km/s")
print(f"Radio final alternativo: {r_final_alt:.4f} km")
print(f"Radio objetivo GEO: {r2:.4f} km")
print(f"Error absoluto: {error_radio_alt:.4f} km")
print(f"Error relativo: {error_relativo_alt:.4f} %")
print(f"Posición final alternativa: x = {x_alt[-1]:.4f} km, y = {y_alt[-1]:.4f} km")

In [ ]:
# 9. Comparación entre Hohmann ideal y trayectoria alternativa

plt.figure(figsize=(8, 8))

plt.plot(x_leo, y_leo, label="Órbita inicial LEO")
plt.plot(x_geo, y_geo, label="Órbita final GEO")
plt.plot(x_num, y_num, linewidth=2.5, label="Hohmann ideal")
plt.plot(x_alt, y_alt, "--", linewidth=2, label="Trayectoria alternativa")

tierra = plt.Circle((0, 0), R_T, fill=False, linewidth=1.5, label="Tierra")
plt.gca().add_patch(tierra)

plt.scatter([r1], [0], s=60, label="Periapsis")
plt.scatter([x_alt[-1]], [y_alt[-1]], s=60, label="Final alternativo")

plt.gca().set_aspect("equal", adjustable="box")
plt.xlabel("x (km)")
plt.ylabel("y (km)")
plt.title("Sensibilidad de la transferencia frente al impulso inicial")
plt.grid(True, alpha=0.3)
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()